# Initialize Hail (run this on start up)

In [ ]:
import hail as hl
import pandas as pd
import numpy as np
import hail as hl
from bokeh.io import show, output_notebook
from bokeh.layouts import gridplot

hl.init(default_reference='GRCh38', idempotent=True)

# Initialize Matrices


Fill in the path to the filtered annotated matrix table in GCS. The path should look somethign like this `gs://../filt_annotated_mt/annotated_combined.mt/`

In [ ]:
# 50k sample filtered annotated VCF matrix table 
filt_annotated_mt_50k = hl.read_matrix_table('')

# Plots

Plots from previously validated data are available [here](https://docs.google.com/presentation/d/1B7G2Qw13__131qtweXhTvtqmMPYJoMLumKiUNE0on6A/edit?usp=sharing) for comparison 

## Histogram of variants per sample

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_variants_per_sample(mt):
    # 1. Calculate number of variants per sample using Allele Depth (AD)
    # We count a variant if the alternate allele depth (AD[1]) is > 0.
    # Note: 'AD' is missing? check safeguards against missing data.
    mt = mt.annotate_cols(
        n_variants = hl.agg.count_where(mt.AD[1] > 0)
    )

    # 2. Convert to Pandas
    df = mt.cols().select('n_variants').to_pandas()

    # 3. Plot
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df, x='n_variants', bins=50, kde=False, color='#4C72B0')
    
    plt.title('Distribution of Variants per Sample (Based on AD > 0)', fontsize=15)
    plt.xlabel('Number of Variants (Alt Depth > 0)', fontsize=12)
    plt.ylabel('Count of Samples', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

# Run the code
plot_variants_per_sample(filt_annotated_mt_50k)

## Mitochondrial Copy Number Distribution

In [ ]:
def plot_mito_cn_distribution(mt):
    # Select the copy number field
    # Note: We handle missing values (drop_na) to prevent plotting errors
    df = mt.cols().select('mito_cn').to_pandas().dropna()

    plt.figure(figsize=(10, 6))
    sns.histplot(df['mito_cn'], bins=50, color='#55A868')
    
    plt.title('Distribution of Mitochondrial Copy Number', fontsize=15)
    plt.xlabel('Mitochondrial Copy Number', fontsize=12)
    plt.ylabel('Count of Samples', fontsize=12)
    
    # Add a vertical line for the median
    median_cn = df['mito_cn'].median()
    plt.axvline(median_cn, color='red', linestyle='--', label=f'Median: {median_cn:.1f}')
    plt.legend()
    plt.show()   
    
# run the code 
plot_mito_cn_distribution(filt_annotated_mt_50k)

From the Gnomad paper:
Histogram shows mtDNA copy number per cell (2 × mean mtDNA coverage/ median nDNA coverage) for all samples, and overlaid histograms show three selected cohorts (223 outliers with mtCN 1250–7000 excluded). Only samples with mtCN 50–500 (dashed lines) were included in the released mtDNA call set (56,434/70,375).

Additionally, ChatGPT thought sex differences (or age) might explain:
• XX samples → higher mtCN (peak around 150–300)
• XY samples → lower mtCN (peak around 50–120)

## Heteroplasmy Level (HL) Histogram

In [ ]:
def plot_heteroplasmy_distribution(mt):
    # 1. Filter to only non-reference calls (we only care about HL for actual variants)
    mt_vars = mt.filter_entries(mt.GT.is_non_ref())
    
    # 2. Create a histogram using Hail's native aggregator
    # Range 0.0 to 1.0 (0% to 100%), with 50 bins
    hist_data = mt_vars.aggregate_entries(hl.agg.hist(mt_vars.HL, 0, 1, 50))
    
    # 3. Extract bin edges and values for plotting
    bin_edges = hist_data.bin_edges
    bin_freq = hist_data.bin_freq
    
    # 4. Plot using Matplotlib bar chart to reconstruct the histogram
    plt.figure(figsize=(12, 6))
    plt.bar(bin_edges[:-1], bin_freq, width=0.02, align='edge', color='#C44E52', edgecolor='black')
    
    plt.title('Distribution of Heteroplasmy Levels (HL) for Non-Ref Variants', fontsize=15)
    plt.xlabel('Heteroplasmy Fraction (0.0 - 1.0)', fontsize=12)
    plt.ylabel('Count of Variants', fontsize=12)
    plt.yscale('log') # Log scale is often helpful for HL plots as low-HL variants dominate
    plt.show()
    
# run the code 
plot_heteroplasmy_distribution(filt_annotated_mt_50k)

## Allele Frequency as within Sample Allele Fraction (Variant Allele Fraction)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


mt = filt_annotated_mt_50k.annotate_entries(
    AF = hl.if_else(
        hl.len(filt_annotated_mt_50k.AD) == 2,
        filt_annotated_mt_50k.AD[1] / hl.sum(filt_annotated_mt_50k.AD),
        hl.missing(hl.tfloat64)
    )
)

In [ ]:
df = (
    mt.entries()
      .select('AF')
      .to_pandas()
      .dropna()
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df['AF'], bins=50)
plt.xlabel("Allele Fraction (VAF)")
plt.ylabel("Count")
plt.title("Distribution of Mitochondrial Allele Fraction")
plt.tight_layout()
plt.show()


## Allele Frequency using AC/AN AF (diploid style)

In [ ]:
import hail as hl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_variant_allele_frequency(mt):
    # 1. Annotate rows with calculated Allele Frequency (AF)
    # Formula: (Count of Hets + Count of Homs) / Total Allele Number
    mt = mt.annotate_rows(
        calculated_AF = (mt.AC_het + mt.AC_hom) / mt.AN
    )

    # 2. Filter to variants where AN > 0 to avoid division errors
    mt = mt.filter_rows(mt.AN > 0)

    # 3. Extract the AF data to Pandas
    df = mt.rows().select('calculated_AF').to_pandas()

    # 4. Plot
    plt.figure(figsize=(10, 6))
    
    # We often use a log scale for AF because rare variants dominate
    sns.histplot(df['calculated_AF'], bins=50, color='#8172B3', log_scale=(False, True))
    
    plt.title('Distribution of Variant Allele Frequency (Population Level)', fontsize=15)
    plt.xlabel('Allele Frequency (AF)', fontsize=12)
    plt.ylabel('Count of Variants (Log Scale)', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()

# run the code:
plot_variant_allele_frequency(filt_annotated_mt_50k)

Notes from Laura:
We altered the code above to remove the 2 (used for diploid samples)

## Allele frequency and allele fraction

In [ ]:
import hail as hl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_variant_AF_and_allele_fraction(mt):
    # --------------------------
    # 1. Population-level AF
    # --------------------------
    mt = mt.annotate_rows(
        calculated_AF = (mt.AC_het + mt.AC_hom) / mt.AN
    )
    mt = mt.filter_rows(mt.AN > 0)
    
    # Use Hail histogram instead of pandas
    af_hist = mt.aggregate_rows(
        hl.agg.hist(mt.calculated_AF, 0, 1, 50)
    )
    
    # --------------------------
    # 2. Per-sample Allele Fraction
    # --------------------------
    mt = mt.annotate_entries(
        allele_fraction = hl.cond(
            (mt.AD[0] + mt.AD[1]) > 0,
            mt.AD[1] / (mt.AD[0] + mt.AD[1]),
            hl.null(hl.tfloat)
        )
    )
    
    # Use Hail histogram for allele fractions instead of collecting all data
    frac_hist = mt.aggregate_entries(
        hl.agg.hist(mt.allele_fraction, 0, 1, 50)
    )
    
    # --------------------------
    # 3. PLOTTING
    # --------------------------
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    
    # --- AF histogram ---
    # Convert Hail histogram to plottable format
    af_bin_edges = af_hist.bin_edges
    af_bin_freq = af_hist.bin_freq
    af_bin_centers = [(af_bin_edges[i] + af_bin_edges[i+1]) / 2 
                      for i in range(len(af_bin_edges) - 1)]
    
    ax[0].bar(af_bin_centers, af_bin_freq, 
              width=(af_bin_edges[1] - af_bin_edges[0]), 
              color='#8172B3', alpha=0.7, edgecolor='black')
    ax[0].set_yscale('log')
    ax[0].set_title('Variant Allele Frequency (Population-Level)', fontsize=15)
    ax[0].set_xlabel('Allele Frequency')
    ax[0].set_ylabel('Count (log)')
    ax[0].grid(axis='y', linestyle='--', alpha=0.5)
    
    # --- Allele fraction histogram ---
    frac_bin_edges = frac_hist.bin_edges
    frac_bin_freq = frac_hist.bin_freq
    frac_bin_centers = [(frac_bin_edges[i] + frac_bin_edges[i+1]) / 2 
                        for i in range(len(frac_bin_edges) - 1)]
    
    ax[1].bar(frac_bin_centers, frac_bin_freq, 
              width=(frac_bin_edges[1] - frac_bin_edges[0]), 
              color='#55A868', alpha=0.7, edgecolor='black')
    ax[1].set_title('Allele Fraction (Per Sample Heteroplasmy)', fontsize=15)
    ax[1].set_xlabel('Allele Fraction (AD1 / DP)')
    ax[1].set_ylabel('Count')
    ax[1].set_xlim(0, 1)
    ax[1].grid(axis='y', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)
    print(f"\nAllele Frequency (Population-level):")
    print(f"  Total variants: {sum(af_hist.bin_freq):,}")
    print(f"  Mean AF: {sum(af_bin_centers[i] * af_bin_freq[i] for i in range(len(af_bin_freq))) / sum(af_bin_freq):.6f}")
    
    print(f"\nAllele Fraction (Per-sample):")
    print(f"  Total entries: {sum(frac_hist.bin_freq):,}")
    print(f"  Mean fraction: {sum(frac_bin_centers[i] * frac_bin_freq[i] for i in range(len(frac_bin_freq))) / sum(frac_bin_freq):.6f}")

# Run the function
plot_variant_AF_and_allele_fraction(filt_annotated_mt_50k)

In [ ]:
mt = filt_annotated_mt_50k

# Calculate allele fraction
mt = mt.annotate_entries(
    allele_fraction = hl.cond(
        hl.sum(mt.AD) > 0,
        mt.AD[1] / hl.sum(mt.AD),
        hl.null(hl.tfloat)
    )
)


## Heteroplasmies by mtCN

#### Define heteroplasmies (0.01–0.50) and NUMT-FPs


putative heteroplasmies: 0.01–0.50 and NOT common_low_heteroplasmy

NUMT-FPs: 0.01–0.50 and common_low_heteroplasmy == True
(acting as your NUMT proxy)

In [ ]:

het_expr = (
    (mt.allele_fraction >= 0.01) &
    (mt.allele_fraction <= 0.50)
)


In [ ]:
numt_expr = het_expr & mt.common_low_heteroplasmy


In [ ]:
mt = mt.annotate_cols(
    num_hets = hl.agg.count_where(het_expr & (~mt.common_low_heteroplasmy)),
    num_numt_fp = hl.agg.count_where(numt_expr)
)


In [ ]:
mt = mt.annotate_cols(
    mtcn_bin = (
        hl.case()
        .when(mt.mito_cn < 25, "0–25")
        .when(mt.mito_cn < 50, "25–50")
        .when(mt.mito_cn < 75, "50–75")
        .when(mt.mito_cn < 100, "75–100")
        .when(mt.mito_cn < 125, "100–125")
        .when(mt.mito_cn < 150, "125–150")
        .when(mt.mito_cn < 175, "150–175")
        .when(mt.mito_cn < 200, "175–200")
        .when(mt.mito_cn < 225, "200–225")
        .when(mt.mito_cn < 250, "225–250")
        .when(mt.mito_cn < 275, "250–275")
        .when(mt.mito_cn < 300, "275–300")
        .default(">=300")
    )
)


In [ ]:
ht_samples = mt.cols().select("mtcn_bin", "num_hets", "num_numt_fp")
df = ht_samples.to_pandas()

# Drop samples with missing mtCN bin if any
df = df.dropna(subset=["mtcn_bin"])


In [ ]:
df_A = (
    df.groupby("mtcn_bin")["num_hets"]
      .mean()
      .reset_index(name="mean_hets")
)
# enforce bin order for prettier x-axis
bin_order = [
    "50–75",
    "75–100",
    "100–125",
    "125–150",
    "150–175",
    "175–200",
    "200–225",
    "225–250",
    "250–275",
    "275–300",
    ">=300"
]

df_A["mtcn_bin"] = pd.Categorical(df_A["mtcn_bin"], categories=bin_order, ordered=True)
df_A = df_A.sort_values("mtcn_bin")


In [ ]:
df_B = (
    df.groupby("mtcn_bin")["num_numt_fp"]
      .mean()
      .reset_index(name="mean_numt_fp")
)
df_B["mtcn_bin"] = pd.Categorical(df_B["mtcn_bin"], categories=bin_order, ordered=True)
df_B = df_B.sort_values("mtcn_bin")


In [ ]:
mt.aggregate_cols(
    hl.agg.collect_as_set(mt.mtcn_bin)
)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
sns.lineplot(data=df_A, x="mtcn_bin", y="mean_hets", marker="o")
plt.title("Mean Putative Heteroplasmies (0.01–0.50) by mtCN")
plt.xlabel("mtDNA Copy Number Bin")
plt.ylabel("Mean Number of Heteroplasmies")
plt.tight_layout()
plt.show()


## NUMT False Positives

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(data=df_B, x="mtcn_bin", y="mean_numt_fp", marker="o")
plt.title("Mean NUMT False Positives (0.01–0.50) by mtCN")
plt.xlabel("mtDNA Copy Number Bin")
plt.ylabel("Mean Number of NUMT-FPs")
plt.tight_layout()
plt.show()


## Plot for haplogroups Supplemental Figure 6 G

In [ ]:
mt = filt_annotated_mt_50k
mt = mt.annotate_entries(
    VAF = hl.if_else(
        hl.len(mt.AD) > 1,
        mt.AD[1] / hl.sum(mt.AD),
        hl.null(hl.tfloat)
    )
)



In [ ]:
ht = mt.cols()

ht.aggregate(
    hl.len(hl.agg.collect_as_set(ht.major_haplogroup))
)

(
    ht.select(ht.major_haplogroup)
      .distinct()
      .show(20)
)




In [ ]:
mt = mt.annotate_cols(
    macro_haplogroup = mt.major_haplogroup[0]
)

In [ ]:
mt = mt.annotate_entries(
    is_heteroplasmic = (
        (mt.VAF >= 0.10) &
        (mt.VAF <= 0.95)
    )
)



In [ ]:
mt = mt.annotate_cols(
    n_het_snvs = hl.agg.count_where(mt.is_heteroplasmic)
)


In [ ]:
list(mt.col)

In [ ]:
mt.aggregate_cols(hl.agg.counter(mt.macro_haplogroup))


In [ ]:
mt.aggregate_cols(hl.agg.counter(mt.hap))


In [ ]:
counts = mt.aggregate_cols(hl.agg.counter(mt.major_haplogroup))


In [ ]:
l_counts = {k: v for k, v in counts.items() if k and k.startswith("L")}


In [ ]:
l_counts

In [ ]:
df = (
    mt.cols()
      .select('hap', 'n_het_snvs')
      .to_pandas()
      .dropna(subset=['hap'])
)


In [ ]:
def haplogroup_origin(hg):
    if hg.startswith('L'):
        return 'African'
    elif hg.startswith(('M', 'D', 'C', 'A', 'B', 'F')):
        return 'Asian'
    else:
        return 'European'

df['origin'] = df['hap'].apply(haplogroup_origin)


In [ ]:
origin_colors = {
    'African': 'purple',
    'Asian': 'green',
    'European': 'blue'
}


In [ ]:
hap_order = (
    df.groupby('hap')#['n_het_snvs']
      #.median()
      .sort_values()
      .index
)


In [ ]:
hap_order = sorted(df['hap'].unique())

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 5))

ax = sns.boxplot(
    data=df,
    x='hap',
    y='n_het_snvs',
    hue='origin',
    order=hap_order,
    palette=origin_colors,
    dodge=False,
    showfliers=False,
    linewidth=0.8
)

ax.set_xlabel("mtDNA haplogroup")
ax.set_ylabel("Heteroplasmic SNVs per sample (VAF 0.10–0.95)")
ax.set_title("Heteroplasmic variants by mtDNA haplogroup")

plt.xticks(rotation=90)
plt.legend(title="Phylogenetic origin")
plt.tight_layout()
plt.show()


## Haplogroups and homoplasmy

In [ ]:
mt = mt.annotate_entries(
    is_homoplasmic = mt.VAF >= 0.95
)


In [ ]:
mt = mt.annotate_cols(
    n_hom_snvs = hl.agg.count_where(mt.is_homoplasmic)
)


In [ ]:
df_hom = (
    mt.cols()
      .select('hap', 'n_hom_snvs')
      .to_pandas()
      .dropna(subset=['hap'])
)


In [ ]:
def haplogroup_origin(hg):
    if hg.startswith('L'):
        return 'African'
    elif hg.startswith(('M', 'D', 'C', 'A', 'B', 'F')):
        return 'Asian'
    else:
        return 'European'

df_hom['origin'] = df_hom['hap'].apply(haplogroup_origin)

In [ ]:
origin_colors = {
    'African': 'purple',
    'Asian': 'green',
    'European': 'blue'
}


In [ ]:
hap_order = sorted(df['hap'].unique())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

ax = sns.boxplot(
    data=df_hom,
    x='hap',
    y='n_hom_snvs',
    order=hap_order,
    hue='origin',
    showfliers=False,
    linewidth=0.8
)

ax.set_xlabel("mtDNA hap")
ax.set_ylabel("Homoplasmic SNVs per sample (VAF ≥ 0.95)")
ax.set_title("Homoplasmic variant burden by mtDNA haplogroup")

plt.tight_layout()
plt.show()
